# 1. Carga de datos procesados y sucursales

Este notebook parte de los datasets ya limpios y con `precio_efectivo` calculado en la etapa anterior (ver `01_analisis_dia_carrefour.ipynb`).  
En esta fase se trabaja sobre:

- los **productos comunes** entre DIA y Carrefour, ya procesados y guardados en formato Parquet;
- los **datasets de sucursales** de cada cadena, que permiten incorporar información geográfica y organizacional (bandera, provincia, localidad, etc.).

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# Cargamos los datos procesados de productos de DIA y productos de Carrefour
productos_dia_comunes = pd.read_parquet('../data/processed/productos_dia_procesado.parquet')
productos_carrefour_comunes = pd.read_parquet('../data/processed/productos_carrefour_procesado.parquet')

In [8]:
# Cargamos los datasets como texto para conservar el formato original de
# identificadores y evitar conversiones automáticas de tipos.
sucursales_dia = pd.read_csv(
    '../data/raw/sucursales_dia.csv',
    delimiter='|',
    encoding='UTF-8',
    dtype=str,
    low_memory=False
)

sucursales_carrefour = pd.read_csv(
    '../data/raw/sucursales_carrefour.csv',
    delimiter='|',
    encoding='UTF-8',
    dtype=str,
    low_memory=False
)

In [9]:
# Comprobamos las dimensiones iniciales de ambos datasets.
print("Dimensiones de sucursales_dia:", sucursales_dia.shape)
print("Dimensiones de sucursales_carrefour:", sucursales_carrefour.shape)

Dimensiones de sucursales_dia: (984, 21)
Dimensiones de sucursales_carrefour: (731, 21)


Tras la carga inicial se inspeccionan las primeras y últimas filas de cada dataset, lo que permite:

- verificar la estructura de las columnas;
- confirmar la presencia de la fila de metadatos al final del archivo (similar a la observada en los datasets de productos);

In [10]:
# Mostramos las primeras y últimas filas de ambos datasets para inspección visual.
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas

print("Primeras 5 filas de sucursales_dia:")
display(sucursales_dia.head(5))

print("Últimas 5 filas de sucursales_dia:")
display(sucursales_dia.tail(5))

print("Primeras 10 filas de sucursales_carrefour:")
display(sucursales_carrefour.head(5))

print("Últimas 10 filas de sucursales_carrefour:")
display(sucursales_carrefour.tail(5))

Primeras 5 filas de sucursales_dia:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
0,15,1,1,1 - VILLA DEL PARQUE,Autoservicio,CL BEIRO,3146,-34.5971373,-58.4976429,NaN,VILLA DEL PARQUE,00001419,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
1,15,1,3,3 - BALVANERA,Autoservicio,CL ENTRE RIOS,735,-34.6173202,-58.3916913,NaN,BALVANERA,00001080,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
2,15,1,4,4 - PARQUE PATRICIOS,Autoservicio,CL CASEROS,2938,-34.6369605,-58.4045551,NaN,PARQUE PATRICIOS,00001264,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
3,15,1,5,5 - SAN CRISTÓBAL,Autoservicio,AV SAN JUAN,2641,-34.6236734,-58.4014697,NaN,SAN CRISTÓBAL,00001232,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
4,15,1,6,6 - FLORES,Autoservicio,CL ARTIGAS,26,-34.6289482,-58.4642662,NaN,FLORES,00001406,CAPITAL FEDERAL,AR-C,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00


Últimas 5 filas de sucursales_dia:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
979,15,1,8014,8014 - METAN,Autoservicio,JOSE IGNACIO SIERRA,126,-25.4987985,-64.9705319,NaN,METAN,00004440,SALTA,AR-A,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
980,15,1,8017,8017 - GRAL. GUEMES,Autoservicio,CABRED,23,-24.671542,-65.045994,NaN,GRAL. GUEMES,00004430,SALTA,AR-A,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
981,15,1,8018,8018 - JUJUY,Autoservicio,BELGRANO,620,-24.387603,-65.264866,NaN,JUJUY,00004603,JUJUY,AR-J,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00,8:00 a 22:00
982,15,1,90000,eCommerce,Supermercado,Juan Francisco Segui,4646,-34.5736049,-58.4232349,NaN,NaN,00001425,CABA,Buenos Aires,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00,00:00 a 24:00
983,Última actualización: 2026-07-03T04:05:37-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Primeras 10 filas de sucursales_carrefour:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
0,10,3,365,Nazca 1944,Autoservicio,Nazca,1944,-34.612145,-58.480997,NaN,Santa Rita,1416,Ciudad Autónoma de Buenos Aires,AR-C,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,10:30 a 20:00
1,10,3,408,Perú 916,Autoservicio,Perú,916,-34.618745,-58.374322,NaN,San telmo,1068,Ciudad Autónoma de Buenos Aires,AR-C,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,07:00 a 20:00,11:00 a 20:00
2,10,4,504,Maxi Cacheuta Cba (504),Hipermercado,Cacheuta 4419,4419,-31.460585,-64.215363,NaN,NaN,5016,Cacheuta,AR-C,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00,08:00 A 22:00
3,10,3,755,C.Tribulato 1279 (San Miguel),Autoservicio,C Tribulato,1279,-34.541507,-58.713798,NaN,NaN,1663,San Miguel,AR-B,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,09:00 A 20:00
4,10,3,479,Avenida Eva Perón 4215,Autoservicio,Avenida Eva Perón,4215,-34.690047,-58.689142,NaN,NaN,1722,Merlo,AR-B,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,10:00 a 19:00


Últimas 10 filas de sucursales_carrefour:


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
726,10,3,344,M. T. de Alvear 2251,Autoservicio,Marcelo T. de Alvear,2251,-34.596862,-58.399571,NaN,Recoleta,1122,Ciudad Autónoma de Buenos Aires,AR-C,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,11:00 a 20:00
727,10,3,593,Entre Ríos 1192,Autoservicio,Entre Ríos,1192,-34.622603,-58.391341,NaN,San Cristobal,1080,Ciudad Autónoma de Buenos Aires,AR-C,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,07:30 a 20:00,11:00 a 20:00
728,10,3,791,BEIRO 3342,Autoservicio,BEIRO,3345,-34.598084,-58.500119,NaN,NaN,1419,Ciudad Autónoma de Buenos Aires,AR-C,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,07:00 A 22:00,11:00 A 20:00
729,10,3,608,Serrano 280,Autoservicio,Serrano,280,-34.598185,-58.444983,NaN,Villa Crespo,1414,Ciudad Autónoma de Buenos Aires,AR-C,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,08:00 a 20:00,11:00 a 20:00
730,Ultima actualización: 2026-07-03T08:30:05-03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


La inspección inicial de los datasets de sucursales muestra que la columna `sucursales_provincia` utiliza códigos ISO 3166‑2, el estándar internacional para identificar subdivisiones dentro de un país. En este formato, la parte posterior al guion corresponde a la provincia dentro de Argentina, por lo que esta columna resulta fundamental para uno de los objetivos centrales del análisis: comparar precios por provincia.

Durante la revisión también se detectó una inconsistencia en una de las filas, donde aparece el nombre “Buenos Aires” en lugar del código ISO correspondiente. Este tipo de irregularidades requiere normalización para asegurar que todas las provincias estén correctamente codificadas y que los análisis posteriores —especialmente los que involucran comparaciones regionales— se basen en información homogénea y confiable.

Además, se observa una fila final de metadatos similar a la presente en los datasets de productos. Esta fila no representa una sucursal real y se elimina antes de continuar, de modo que el análisis posterior se realice únicamente sobre registros válidos.

In [11]:
# Eliminamos filas con valores nulos en la columna 'id_sucursal' para ambos datasets
sucursales_dia = sucursales_dia.dropna(subset=['id_sucursal']).copy()
sucursales_carrefour = sucursales_carrefour.dropna(subset=['id_sucursal']).copy()

In [12]:
# Inspeccionar nulos en ambos datasets.
print("\nNulos en el dataset de Sucursales DIA:")
print("\n",sucursales_dia.isna().sum())
print("\nNulos en el dataset de Sucursales Carrefour:")
print("\n", sucursales_carrefour.isna().sum())


Nulos en el dataset de Sucursales DIA:

 id_comercio                                0
id_bandera                                 0
id_sucursal                                0
sucursales_nombre                          0
sucursales_tipo                            0
sucursales_calle                           0
sucursales_numero                        429
sucursales_latitud                         0
sucursales_longitud                        0
sucursales_observaciones                 983
sucursales_barrio                          1
sucursales_codigo_postal                   0
sucursales_localidad                       0
sucursales_provincia                       0
sucursales_lunes_horario_atencion          0
sucursales_martes_horario_atencion         0
sucursales_miercoles_horario_atencion      0
sucursales_jueves_horario_atencion         0
sucursales_viernes_horario_atencion        0
sucursales_sabado_horario_atencion         0
sucursales_domingo_horario_atencion        0
dtype: int64


El análisis de valores nulos muestra dos situaciones distintas entre las cadenas. En el caso de DIA, los nulos se concentran en campos no críticos como `sucursales_numero`, `sucursales_observaciones` y, en menor medida, `sucursales_barrio`, sin afectar información estructural relevante para el análisis geográfico.

En Carrefour, en cambio, se observan **36 nulos en `sucursales_provincia`**, y esa misma cantidad se repite en varias columnas asociadas a horarios de atención. Este patrón indica que existen **filas incompletas**, no simples valores faltantes aislados. Dado que la provincia es una variable central para el análisis regional, será necesario identificar y corregir estos registros antes de avanzar con las comparaciones por provincia.

In [13]:
# Convertimos las columnas de latitud y longitud a tipo numérico, 
# manejando errores con 'coerce' para convertir valores no numéricos a NaN.
sucursales_carrefour['sucursales_latitud'] = pd.to_numeric(
    sucursales_carrefour['sucursales_latitud'], errors='coerce'
)

sucursales_carrefour['sucursales_longitud'] = pd.to_numeric(
    sucursales_carrefour['sucursales_longitud'], errors='coerce'
)

In [14]:
carrefour_prov_nulos = sucursales_carrefour[sucursales_carrefour['sucursales_provincia'].isna()].copy()
carrefour_prov_nulos


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion
32,10,3,6018,"Viamonte 5366, Luján de Cuyo, Mendoza",Autoservicio,Viamonte 5366,5366,-32.986263,-68.882983,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,10,3,6009,"Balcarce 401, Ciudad de Salta",Autoservicio,Balcarce 401,401,-24.784593,-65.412082,NaN,NaN,4400,SALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,10,3,6016,"Boedo 2879, Luján de Cuyo, Mendoza",Autoservicio,Boedo 2879,2879,-32.987800,-68.829942,NaN,NaN,5503,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132,10,3,6023,"Av San Martin 792, Ciudad de Mendoza",Autoservicio,Av San Martin 792,792,-32.893272,-68.840495,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133,10,3,6025,"Aristides 287, Ciudad de Mendoza",Autoservicio,Aristides 287,287,-32.891987,-68.854527,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138,10,3,6022,"Hipolito Yrigoyen 2695, Maipú, Mendoza (Comple...",Autoservicio,Hipolito Yirigoyen 2695,2695,-32.959835,-68.783645,NaN,NaN,5515,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,10,3,6005,MINI - RIVADEO 1487,Autoservicio,Rivadeo 1487,1487,-31.400816,-64.181451,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168,10,3,6032,"Avenida Boyacá 613, Capital Federal",Autoservicio,Avenida Boyacá 613,613,-34.620054,-58.461094,NaN,NaN,1406,Ciudad Autónoma de Buenos Aires,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,10,3,6007,MINI - J B ALBERDI 3099,Autoservicio,Juan Bautista Alberdi 3099,3099,-34.633684,-58.471208,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
225,10,3,6012,"Avenida Bordereau 9950, Villa Allende, Córdoba...",Autoservicio,Avenida Bordereau 9950,9950,-31.309289,-64.287994,NaN,NaN,5105,CORDOBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


La revisión de los valores faltantes muestra que la mayoría de los registros con `sucursales_provincia` nulo pueden recuperarse utilizando la información disponible en `sucursales_localidad`, donde en muchos casos aparece directamente el nombre de la provincia. En otros registros, la localidad corresponde a una ciudad, lo que permite deducir la provincia de forma confiable. Para los pocos casos restantes, la provincia puede inferirse a partir del nombre de la sucursal o, en última instancia, mediante la ubicación geográfica utilizando `sucursales_latitud` y `sucursales_longitud`. De este modo, es posible completar los 36 registros incompletos antes de continuar con el análisis regional.

In [15]:
# Vemos las columnas relevantes de las sucursales de Carrefour que tienen valores nulos en la columna 'sucursales_provincia'.
carrefour_prov_nulos[['id_sucursal', 'sucursales_nombre', 'sucursales_localidad',
                      'sucursales_latitud', 'sucursales_longitud']]

,id_sucursal,sucursales_nombre,sucursales_localidad,sucursales_latitud,sucursales_longitud
32,6018,"Viamonte 5366, Luján de Cuyo, Mendoza",Mendoza,-32.986263,-68.882983
62,6009,"Balcarce 401, Ciudad de Salta",SALTA,-24.784593,-65.412082
72,6016,"Boedo 2879, Luján de Cuyo, Mendoza",Mendoza,-32.987800,-68.829942
132,6023,"Av San Martin 792, Ciudad de Mendoza",Mendoza,-32.893272,-68.840495
133,6025,"Aristides 287, Ciudad de Mendoza",Mendoza,-32.891987,-68.854527
138,6022,"Hipolito Yrigoyen 2695, Maipú, Mendoza (Comple...",Mendoza,-32.959835,-68.783645
151,6005,MINI - RIVADEO 1487,-,-31.400816,-64.181451
168,6032,"Avenida Boyacá 613, Capital Federal",Ciudad Autónoma de Buenos Aires,-34.620054,-58.461094
181,6007,MINI - J B ALBERDI 3099,-,-34.633684,-58.471208
225,6012,"Avenida Bordereau 9950, Villa Allende, Córdoba...",CORDOBA,-31.309289,-64.287994


In [16]:
# Identificamos las sucursales de Carrefour donde la localidad coincide con el nombre de la provincia.
# Para ello, creamos una lista de provincias argentinas (sin tildes)
# y comparamos con la columna 'sucursales_localidad' del dataset de Carrefour.
provincias_arg = [
    'Buenos Aires', 'CABA', 'Catamarca', 'Chaco', 'Chubut', 'Cordoba',
    'Corrientes', 'Entre Rios', 'Formosa', 'Jujuy', 'La Pampa', 'La Rioja',
    'Mendoza', 'Misiones', 'Neuquen', 'Rio Negro', 'Salta', 'San Juan',
    'San Luis', 'Santa Cruz', 'Santa Fe', 'Santiago del Estero',
    'Tierra del Fuego', 'Tucuman'
]

# Normalizamos la columna 'sucursales_localidad' 
carrefour_prov_nulos['sucursales_localidad_norm'] = (
    carrefour_prov_nulos['sucursales_localidad']
    .str.upper()
    .str.strip()
)
# Normalizamos la lista de provincias para compararlas en mayúsculas
provincias_arg_upper = [p.upper() for p in provincias_arg]

carrefour_localidad_es_provincia = carrefour_prov_nulos[
    carrefour_prov_nulos['sucursales_localidad_norm'].isin(provincias_arg_upper)
]

print(f'Número de sucursales donde la ciudad comparte el nombre de su provincia: {carrefour_localidad_es_provincia["sucursales_localidad"].count()}')

carrefour_localidad_es_provincia


Número de sucursales donde la ciudad comparte el nombre de su provincia: 25


,id_comercio,id_bandera,id_sucursal,sucursales_nombre,sucursales_tipo,sucursales_calle,sucursales_numero,sucursales_latitud,sucursales_longitud,sucursales_observaciones,sucursales_barrio,sucursales_codigo_postal,sucursales_localidad,sucursales_provincia,sucursales_lunes_horario_atencion,sucursales_martes_horario_atencion,sucursales_miercoles_horario_atencion,sucursales_jueves_horario_atencion,sucursales_viernes_horario_atencion,sucursales_sabado_horario_atencion,sucursales_domingo_horario_atencion,sucursales_localidad_norm
32,10,3,6018,"Viamonte 5366, Luján de Cuyo, Mendoza",Autoservicio,Viamonte 5366,5366,-32.986263,-68.882983,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
62,10,3,6009,"Balcarce 401, Ciudad de Salta",Autoservicio,Balcarce 401,401,-24.784593,-65.412082,NaN,NaN,4400,SALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SALTA
72,10,3,6016,"Boedo 2879, Luján de Cuyo, Mendoza",Autoservicio,Boedo 2879,2879,-32.987800,-68.829942,NaN,NaN,5503,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
132,10,3,6023,"Av San Martin 792, Ciudad de Mendoza",Autoservicio,Av San Martin 792,792,-32.893272,-68.840495,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
133,10,3,6025,"Aristides 287, Ciudad de Mendoza",Autoservicio,Aristides 287,287,-32.891987,-68.854527,NaN,NaN,5500,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
138,10,3,6022,"Hipolito Yrigoyen 2695, Maipú, Mendoza (Comple...",Autoservicio,Hipolito Yirigoyen 2695,2695,-32.959835,-68.783645,NaN,NaN,5515,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
225,10,3,6012,"Avenida Bordereau 9950, Villa Allende, Córdoba...",Autoservicio,Avenida Bordereau 9950,9950,-31.309289,-64.287994,NaN,NaN,5105,CORDOBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CORDOBA
243,10,3,6030,"Avenida Castro Barros 858, Córdoba Capital",Autoservicio,Avenida Castro Barros 858,858,-31.397186,-64.199161,NaN,NaN,5000,CORDOBA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CORDOBA
255,10,3,6031,"Italia 5833, Luján de Cuyo, Mendoza",Autoservicio,Italia 5833,5833,-32.982480,-68.878215,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA
290,10,3,6017,"Viamonte 4620, Luján de Cuyo, Mendoza",Autoservicio,Viamonte 4620,4620,-32.992785,-68.887172,NaN,NaN,5505,Mendoza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENDOZA


Del análisis de los 36 registros con `sucursales_provincia` nulo se observa que, en 25 de ellos, la columna `sucursales_localidad` contiene directamente el nombre de la provincia. Tras normalizar el texto (mayúsculas y eliminación de espacios), estas localidades coinciden de manera exacta con los nombres de las provincias argentinas, por lo que estos casos pueden imputarse de forma segura y sin ambigüedad.

Completar estos 25 registros es un paso necesario antes de continuar, ya que en el resto del dataset las provincias aparecen codificadas en formato ISO 3166‑2. Una vez imputados estos valores, será posible reemplazar los códigos ISO por los nombres de las provincias de manera consistente en todo el dataset, asegurando una estructura homogénea para el análisis regional.

In [17]:
# Imputamos la provincia para las 25 sucursales donde la localidad coincide
# con el nombre de la provincia.
sucursales_carrefour.loc[
    carrefour_localidad_es_provincia.index,
    'sucursales_provincia'
] = carrefour_localidad_es_provincia['sucursales_localidad_norm']


In [18]:
# Identificamos las sucursales cuya localidad NO coincide con ninguna provincia argentina
carrefour_localidad_es_ciudad = carrefour_prov_nulos[
    ~carrefour_prov_nulos['sucursales_localidad_norm'].isin(provincias_arg_upper)
]

carrefour_localidad_es_ciudad[['sucursales_localidad', 'sucursales_nombre',
                      'sucursales_latitud', 'sucursales_longitud']]


,sucursales_localidad,sucursales_nombre,sucursales_latitud,sucursales_longitud
151,-,MINI - RIVADEO 1487,-31.400816,-64.181451
168,Ciudad Autónoma de Buenos Aires,"Avenida Boyacá 613, Capital Federal",-34.620054,-58.461094
181,-,MINI - J B ALBERDI 3099,-34.633684,-58.471208
277,-,MINI - LOS GRANADEROS 2269 CBA,-31.377924,-64.204323
330,MAR DEL PLATA,MINI - LURO 3060 (832),-37.995534,-57.550968
355,MAR DEL PLATA,ALBERTI 1511,-38.011862,-57.541844
471,MAR DEL PLATA,MINI - GUEMES 2470 (801),-38.011354,-57.538653
501,-,MINI - SAN JUAN 490 CBA,-31.418132,-64.192626
523,MAR DEL PLATA,MINI - SAN MARTIN 3579 (803),-37.994060,-57.556713
595,MAR DEL PLATA,MINI - SANTA FE 2248 (800),-38.004675,-57.548313


In [19]:
# Diccionario de ciudades para imputar provincia
ciudad_a_provincia = {
    'CIUDAD AUTÓNOMA DE BUENOS AIRES': 'CABA',
    'CIUDAD AUTONOMA DE BUENOS AIRES': 'CABA',
    'MAR DEL PLATA': 'BUENOS AIRES',
}


In [20]:
# Completamos la provincia usando el diccionario de ciudades a provincias.
sucursales_carrefour.loc[
    carrefour_localidad_es_ciudad.index,
    'sucursales_provincia'
] = carrefour_localidad_es_ciudad['sucursales_localidad_norm'].map(ciudad_a_provincia).str.upper()


Las sucursales con localidad '-' se completarán usando rangos geográficos basados en las coordenadas características de la provincia de Córdoba y CABA. 

In [21]:
# Identificamos las sucursales de Carrefour donde la localidad es un guion ('-').
carrefour_localidad_guion = carrefour_localidad_es_ciudad[
    carrefour_localidad_es_ciudad['sucursales_localidad_norm'] == '-'
]

In [22]:
# Imputamos la provincia, según sus coordenadas geográficas,
# para las sucursales de Carrefour donde la localidad es un guion.
cordoba_ubi = (
    (carrefour_localidad_guion['sucursales_latitud'] < -30) &
    (carrefour_localidad_guion['sucursales_latitud'] > -35) &
    (carrefour_localidad_guion['sucursales_longitud'] < -61) &
    (carrefour_localidad_guion['sucursales_longitud'] > -65)
)

sucursales_carrefour.loc[
    carrefour_localidad_guion[cordoba_ubi].index,
    'sucursales_provincia'
] = 'CORDOBA'

caba_ubi = (
    (carrefour_localidad_guion['sucursales_latitud'] < -34) &
    (carrefour_localidad_guion['sucursales_latitud'] > -35) &
    (carrefour_localidad_guion['sucursales_longitud'] < -58) &
    (carrefour_localidad_guion['sucursales_longitud'] > -59)
)

sucursales_carrefour.loc[
    carrefour_localidad_guion[caba_ubi].index,
    'sucursales_provincia'
] = 'CABA'


In [23]:
# Verificamos que no queden provincias sin imputar
sucursales_carrefour['sucursales_provincia'].isna().sum()

np.int64(0)

Las 36 sucursales quedaron completamente imputadas, listas para continuar con la conversión de códigos ISO y el análisis geográfico.